In [9]:
!pip install "Transformers[torch]"


In [10]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd 
from transformers import T5Tokenizer, T5ForConditionalGeneration,Trainer, TrainingArguments

In [12]:
import pandas as pd

train_data = pd.read_csv("/kaggle/input/datasets/sovanbarik/textconversation/samsum-train.csv")
val_data = pd.read_csv("/kaggle/input/datasets/sovanbarik/textconversation/samsum-validation.csv")
test_data = pd.read_csv("/kaggle/input/datasets/sovanbarik/textconversation/samsum-validation.csv")

train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [13]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [14]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [15]:
train_data.sample(10)

,id,dialogue,summary
7998,13731310,Isaac: Bah humbug!\r\nRenee: Oh come on!\r\nIs...,Isaac is hating the Christmas ambiance because...
3179,13727625,Hanna: Plans for Friday evening?\r\nKate: Not ...,Hanna and Kate go for drinks Friday evening ar...
6576,13729663,Jacqueline: hey i have a question\r\nGrant: wh...,Jacqueline is choosing her courses for next se...
1839,13715881,"Steffi: Hi, it's Steffi here :)\r\nGary: Hi St...","Steffi has just become a member of Gary, Fiona..."
11456,13729269-1,"Mari: Hi Dave, how's it going?\r\nDave: Fine, ...",Mari has eaten out with her sister and the kid...
13252,13862827,"Kenneth: Hi Mark! ""Long"" time no speak ;) \nKe...",Kenneth is going to Sweden and wants to top up...
681,13813432,Zara: <file_gif>\r\nZara: something terrible h...,"Zara has lost the earring, which Stanley gave ..."
9644,13730640,Hannah: I’m waiting for you downstairs\r\nFlor...,Hannah is waiting for Florence downstairs and ...
13499,13814915,Emma: Can you please ask mother to cook me mea...,Emma and Noah's mother is not well. Noah asks ...
14182,13829542,Kasia: When are u coming back?\r\nMatt: Back w...,Matt doesn't know when he's coming back to War...


In [16]:
train_data.shape

(14732, 3)

In [17]:
val_data.shape


(818, 3)

In [18]:
# randomly sample rows from training/validation data safely
train_data = train_data.sample(n=min(4000, len(train_data)), random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=min(1000, len(val_data)), random_state=42).reset_index(drop=True)

In [19]:
train_data.shape

(4000, 3)

#Data Cleaning and Preprocessing


In [20]:

import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #lines
    text = re.sub(r"\s+", " ", text) #extra spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower() #strip and lower
    return text

In [21]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [22]:
train_data["summary"][0]

"violet sent claire austin's article."

#Tokenize

In [23]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [24]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(data["summary"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs
val_data_set = val_data.apply(tokenize, axis=1).tolist()

In [25]:
train_data_set = train_data.apply(tokenize, axis=1).tolist()
val_data_set = val_data.apply(tokenize, axis=1).tolist()

In [26]:
train_data_set[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [27]:
len(train_data_set[0]["input_ids"])

512

In [28]:
type(train_data_set)
type(val_data_set)

list

#Working With Our Model


In [29]:
#NLP => generative pre-trained transformer (GPT) => T5 (Text-to-Text Transfer Transformer)
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [30]:

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
model.to(device)

Using device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500,
)

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data_set,
    eval_dataset=val_data_set,
)

In [33]:
# Training the model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,1.140951
2,7.127893,0.859788
3,7.127893,0.829844
4,0.892196,0.815369
5,0.892196,0.810938
6,0.849013,0.809507


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=2.9563673299153646, metrics={'train_runtime': 676.7976, 'train_samples_per_second': 35.461, 'train_steps_per_second': 2.216, 'total_flos': 3248203235328000.0, 'train_loss': 2.9563673299153646, 'epoch': 6.0})

In [34]:
model.save_pretrained("t5-small-samsum")
tokenizer.save_pretrained("t5-small-samsum")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('t5-small-samsum/tokenizer_config.json', 't5-small-samsum/tokenizer.json')

In [35]:
model = T5ForConditionalGeneration.from_pretrained("t5-small-samsum")
tokenizer = T5Tokenizer.from_pretrained("t5-small-samsum")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [44]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.to(device)
    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary

In [45]:
test_dialogue = "A: Hello, how are you? B: I'm good, thanks! How about you? A: I'm doing well. B: That's great to hear. A: What have you been up to lately? B: Just working and spending time with family. A: Sounds nice. B: Yeah, it's been a busy but fulfilling time."

summary = summarize_dialogue(test_dialogue)
print(summary)



b is doing well. b is working and spending time with family.
